# Classificação de Renda — Adult Income Dataset
## Portfólio | Machine Learning

**Objetivo:** Prever se uma pessoa ganha mais ou menos de 50 mil dólares por ano.

**Métrica principal:** F1 Score

---

### Estrutura do projeto
| Etapa | O que fazemos | Por quê |
|-------|--------------|--------|
| 1. EDA | Explorar os dados | Entender o problema antes de modelar |
| 2. Pré-processamento | Tratar NaNs e outliers | Dados sujos quebram qualquer modelo |
| 3. Feature Engineering | Codificar e escalar | Modelos só entendem números |
| 4. Modelagem | Treinar e comparar | Encontrar o melhor algoritmo |
| 5. Tuning | Otimizar hiperparâmetros | Extrair o máximo do modelo escolhido |
| 6. XAI | Explicar o modelo | Saber *por que* ele decide assim |
| 7. Avaliação final | Testar no test.csv | Resultado real, sem viés |

In [ ]:
# =============================================================================
# CÉLULA 1 — INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS
# O que faz : instala pacotes ausentes e carrega todas as bibliotecas usadas
#             ao longo do notebook.
# Resultado : mensagem "Bibliotecas carregadas com sucesso!" sem erros.
# =============================================================================

# %pip instala diretamente no kernel ativo do Jupyter (evita o problema de
# instalar no Python errado que acontece com subprocess).
# -q = modo silencioso (sem logs extensos)
%pip install shap lightgbm xgboost -q

# Suprime avisos de depreciação para manter o output limpo
import warnings
warnings.filterwarnings('ignore')

# ── Manipulação e análise de dados ───────────────────────────────────────────
import pandas as pd          # DataFrames: leitura, filtragem, agregação
import numpy as np           # Operações numéricas vetorizadas

# ── Visualização ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt   # Gráficos base (histogramas, barras, etc.)
import seaborn as sns             # Gráficos estatísticos (heatmap, etc.)

# ── Pré-processamento ─────────────────────────────────────────────────────────
from sklearn.preprocessing import (
    LabelEncoder,    # Converte categorias (string) em inteiros
    StandardScaler   # Normaliza features para média=0 e desvio=1
)

# ── Seleção de features ───────────────────────────────────────────────────────
from sklearn.feature_selection import (
    SelectKBest,            # Seleciona as K melhores features
    mutual_info_classif     # Critério: informação mútua com o target
)

# ── Algoritmos de classificação ───────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression          # Modelo linear (baseline)
from sklearn.tree import DecisionTreeClassifier              # Árvore de decisão
from sklearn.ensemble import (
    RandomForestClassifier,      # Ensemble de árvores independentes
    GradientBoostingClassifier   # Ensemble sequencial (boosting)
)

# ── Métricas de avaliação ─────────────────────────────────────────────────────
from sklearn.metrics import (
    f1_score,              # Média harmônica entre precisão e recall (MÉTRICA PRINCIPAL)
    accuracy_score,        # % de previsões corretas
    precision_score,       # Dos que previu positivo, quantos eram positivos?
    recall_score,          # Dos positivos reais, quantos o modelo encontrou?
    roc_auc_score,         # Área sob a curva ROC (poder discriminativo geral)
    classification_report, # Tabela completa por classe
    confusion_matrix,      # Matriz de acertos e erros por classe
    ConfusionMatrixDisplay # Visualização da matriz de confusão
)

# ── Otimização de hiperparâmetros ─────────────────────────────────────────────
from sklearn.model_selection import (
    RandomizedSearchCV,  # Busca aleatória no espaço de hiperparâmetros
    cross_val_score      # Validação cruzada k-fold
)

# ── Explicabilidade (XAI) ─────────────────────────────────────────────────────
import shap  # SHapley Additive exPlanations: explica cada previsão individualmente

# ── Reprodutibilidade ─────────────────────────────────────────────────────────
# Fixar a semente garante que resultados aleatórios sejam os mesmos em
# qualquer execução — fundamental para experimentos científicos reproduzíveis.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Bibliotecas carregadas com sucesso!')

---
## 1. Carregamento e Exploração dos Dados (EDA)

**Por que EDA primeiro?**  
Antes de qualquer modelagem, precisamos entender a distribuição das variáveis, identificar padrões e detectar problemas. Um cientista de dados que modela sem explorar os dados primeiro está voando às cegas.

O Adult Income Dataset usa `?` para representar valores ausentes — isso é comum em dados reais.

In [ ]:
# =============================================================================
# CÉLULA 2 — CARREGAMENTO DOS DADOS
# O que faz : lê os três arquivos CSV (treino, validação e teste) e faz uma
#             limpeza inicial nos espaços em branco.
# Resultado : 3 DataFrames carregados com shape e contagem de linhas exibidos.
# =============================================================================

# na_values=' ?' instrui o pandas a tratar ' ?' como NaN automaticamente.
# No Adult Dataset, valores desconhecidos foram codificados com ' ?' no CSV.
train = pd.read_csv('train.csv', na_values=' ?')       # 70% — treina o modelo
val   = pd.read_csv('validation.csv', na_values=' ?')  # 15% — tuning e avaliação
test  = pd.read_csv('test.csv', na_values=' ?')        # 15% — avaliação final APENAS

# Remove espaços extras dos nomes de colunas e dos valores string.
# CSVs do Census costumam ter espaços à esquerda nos valores (ex: ' Private').
for df in [train, val, test]:
    df.columns = df.columns.str.strip()                        # limpa cabeçalhos
    for col in df.select_dtypes('object').columns:
        df[col] = df[col].str.strip()                          # limpa strings

# Exibe o tamanho de cada split para confirmar a proporção 70/15/15
print(f'Train:      {train.shape[0]:,} linhas x {train.shape[1]} colunas')
print(f'Validation: {val.shape[0]:,} linhas x {val.shape[1]} colunas')
print(f'Test:       {test.shape[0]:,} linhas x {test.shape[1]} colunas')

In [ ]:
# =============================================================================
# CÉLULA 3 — VISUALIZAÇÃO DAS PRIMEIRAS LINHAS
# O que faz : exibe as 5 primeiras linhas do conjunto de treino.
# Resultado : tabela com todas as 15 colunas e 5 exemplos reais de registros.
#             Permite verificar tipos de dados, categorias existentes e formato.
# =============================================================================
train.head()

In [ ]:
# =============================================================================
# CÉLULA 4 — INFORMAÇÕES ESTRUTURAIS DO DATASET
# O que faz : exibe o tipo de dado de cada coluna, contagem de não-nulos e
#             uso de memória do DataFrame.
# Resultado : lista de colunas com dtype (int64, object, etc.) e contagem de
#             valores presentes. Colunas com count < total indicam NaNs.
# =============================================================================
train.info()

In [ ]:
# =============================================================================
# CÉLULA 5 — ESTATÍSTICAS DESCRITIVAS
# O que faz : calcula estatísticas resumidas para todas as variáveis numéricas.
# Resultado : tabela com count, mean, std, min, 25%, 50%, 75% e max.
#             Permite identificar outliers (max muito maior que 75%),
#             assimetria (mean muito diferente da mediana) e escala de valores.
# =============================================================================
train.describe()

In [ ]:
# =============================================================================
# CÉLULA 6 — DISTRIBUIÇÃO DA VARIÁVEL-ALVO (TARGET)
# O que faz : conta e exibe em gráfico de barras a proporção de cada classe
#             do target 'income' (<=50K vs >50K).
# Resultado : gráfico salvo em 'fig_target_distribuicao.png' e contagem
#             impressa. Revela o grau de desbalanceamento das classes.
# =============================================================================

# value_counts() conta ocorrências de cada valor único na coluna
target_counts = train['income'].value_counts()

# normalize=True retorna proporção (0.0 a 1.0); multiplicamos por 100 para %
target_pct = train['income'].value_counts(normalize=True) * 100

# Criação do gráfico de barras
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(target_counts.index, target_counts.values, color=['#2196F3', '#FF5722'])
ax.set_title('Distribuição da variável-alvo (income)', fontsize=14)
ax.set_ylabel('Quantidade')

# Adiciona o percentual acima de cada barra para facilitar leitura
for bar, pct in zip(bars, target_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{pct:.1f}%', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_target_distribuicao.png', dpi=120)  # salva em alta resolução
plt.show()

# Imprime os valores absolutos de cada classe
print(target_counts.to_string())

**Observação sobre desbalanceamento:**  
O dataset tem aproximadamente 75% `<=50K` e 25% `>50K`. Isso é desbalanceamento moderado. Por isso usamos **F1 Score** como métrica principal — ela penaliza o modelo que simplesmente "chuta" a classe majoritária. Acurácia aqui seria enganosa.

In [ ]:
# =============================================================================
# CÉLULA 7 — DISTRIBUIÇÃO DAS VARIÁVEIS NUMÉRICAS
# O que faz : plota um histograma para cada variável numérica do dataset.
# Resultado : grade de histogramas salva em 'fig_numericas_distribuicao.png'.
#             Mostra a forma da distribuição (normal, assimétrica, bimodal)
#             e ajuda a identificar outliers visualmente.
# =============================================================================

# Seleciona apenas colunas com tipo numérico (int64, float64)
num_cols = train.select_dtypes(include='number').columns.tolist()

# Cria uma grade 2x4 de subgráficos (uma célula por feature numérica)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()  # transforma a matriz 2D em lista 1D para iterar

for i, col in enumerate(num_cols):
    # bins=40 = número de barras do histograma (mais barras = mais detalhe)
    axes[i].hist(train[col].dropna(), bins=40, color='#2196F3', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequência')

# Oculta subgráficos extras caso haja menos features que células na grade
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das variáveis numéricas', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_numericas_distribuicao.png', dpi=120)
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 8 — DISTRIBUIÇÃO DAS VARIÁVEIS CATEGÓRICAS
# O que faz : plota um gráfico de barras horizontal para cada variável
#             categórica, mostrando as 10 categorias mais frequentes.
# Resultado : grade de gráficos salva em 'fig_categoricas_distribuicao.png'.
#             Revela categorias dominantes e raras (possíveis candidatas a
#             agrupamento ou descarte).
# =============================================================================

# Seleciona colunas de texto (object), excluindo o target 'income'
cat_cols = train.select_dtypes(include='object').columns.drop('income').tolist()

# Grade 3x3 — suficiente para as 9 variáveis categóricas do dataset
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = train[col].value_counts().head(10)  # top 10 categorias por frequência
    axes[i].barh(vc.index, vc.values, color='#4CAF50')
    axes[i].set_title(col)
    axes[i].invert_yaxis()  # coloca a categoria mais frequente no topo

# Oculta subgráficos sem dados
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das variáveis categóricas (top 10)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_categoricas_distribuicao.png', dpi=120)
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 9 — MATRIZ DE CORRELAÇÃO
# O que faz : calcula a correlação de Pearson entre todos os pares de variáveis
#             numéricas e exibe como mapa de calor (heatmap).
# Resultado : heatmap salvo em 'fig_correlacao.png'.
#             Valores próximos de +1 = correlação positiva forte.
#             Valores próximos de -1 = correlação negativa forte.
#             Valores próximos de  0 = sem relação linear.
#             Identifica pares redundantes (multicolinearidade).
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 7))

# .corr() calcula a matriz de correlação de Pearson entre colunas numéricas
corr_matrix = train[num_cols].corr()

# Máscara triangular superior: evita duplicar a informação no heatmap
# (correlação de A com B = correlação de B com A)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# annot=True = exibe o valor numérico em cada célula
# cmap='coolwarm' = azul para negativo, vermelho para positivo
# vmin/vmax = fixa a escala de cores entre -1 e +1
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax)

ax.set_title('Matriz de correlação — variáveis numéricas', fontsize=13)
plt.tight_layout()
plt.savefig('fig_correlacao.png', dpi=120)
plt.show()

---
## 2. Pré-Processamento

### 2.1 Tratamento de Valores Ausentes (NaNs)

**Por que tratar NaNs?**  
A maioria dos algoritmos de ML não aceita valores ausentes — eles causam erros ou comportamentos imprevisíveis. Há três estratégias principais:
- **Remover a linha**: funciona quando são poucos casos (<5%)
- **Imputar pela moda** (categórico) ou **mediana** (numérico): mantém mais dados
- **Criar uma categoria "Desconhecido"**: quando a ausência em si é informativa

Usaremos moda para variáveis categóricas, pois as colunas com NaN neste dataset são categóricas (`workclass`, `occupation`, `native-country`).

In [ ]:
# =============================================================================
# CÉLULA 10 — ANÁLISE DE VALORES AUSENTES (NaNs)
# O que faz : conta os valores nulos em cada coluna dos três splits e calcula
#             o percentual de ausência em relação ao treino.
# Resultado : tabela mostrando apenas colunas com pelo menos 1 NaN, com
#             quantidade por split e % no treino. Indica quais colunas precisam
#             de imputação e se a proporção é aceitável (< 5% = seguro).
# =============================================================================

# isnull() retorna True/False por célula; .sum() soma os True (= contagem de NaN)
nan_report = pd.DataFrame({
    'train': train.isnull().sum(),   # quantidade de NaNs no treino
    'val':   val.isnull().sum(),     # quantidade de NaNs na validação
    'test':  test.isnull().sum()     # quantidade de NaNs no teste
})

# Calcula o percentual de ausência em relação ao total de linhas do treino
nan_report['train_%'] = (nan_report['train'] / len(train) * 100).round(2)

# Exibe apenas colunas que possuem NaN — colunas limpas são ocultadas
print(nan_report[nan_report['train'] > 0])

In [ ]:
# =============================================================================
# CÉLULA 11 — IMPUTAÇÃO DE VALORES AUSENTES (MODA)
# O que faz : substitui NaNs em variáveis categóricas pelo valor mais frequente
#             (moda) de cada coluna, calculado exclusivamente no treino.
# Resultado : mensagem confirmando a moda usada em cada coluna e total de
#             NaNs restantes (deve ser 0). A moda do treino é aplicada também
#             em validação e teste para evitar data leakage.
# =============================================================================

# Identifica colunas categóricas que ainda têm NaN
cat_nan_cols = [c for c in cat_cols if train[c].isnull().any()]

# Calcula a moda SOMENTE no treino.
# mode()[0] retorna o primeiro valor mais frequente (pode haver empate).
# REGRA DE OURO: nunca calcular estatísticas de imputação no val/test —
# isso seria usar informação do "futuro" para preencher o treino.
modes = {col: train[col].mode()[0] for col in cat_nan_cols}

# Aplica a moda calculada no treino para preencher NaNs em todos os splits
for df in [train, val, test]:
    for col, mode_val in modes.items():
        df[col] = df[col].fillna(mode_val)

# Confirma quais valores foram usados na imputação
print('Moda calculada no treino e aplicada em todos os splits:')
for col, mv in modes.items():
    print(f'  {col}: "{mv}"')

# Verifica se todos os NaNs foram tratados (deve imprimir 0)
print(f'\nNaNs restantes no train: {train.isnull().sum().sum()}')

### 2.2 Tratamento de Outliers

**O que é um outlier?**  
Um valor que está muito distante do padrão do conjunto. Pense em capital-gain: a maioria das pessoas tem 0, mas algumas têm 99.999. Isso pode distorcer o modelo.

**Método IQR (Intervalo Interquartil):**  
- Q1 = 25º percentil, Q3 = 75º percentil
- IQR = Q3 - Q1
- Limite inferior = Q1 - 1,5 × IQR
- Limite superior = Q3 + 1,5 × IQR

Valores fora desse intervalo são outliers. Ao invés de remover, vamos usar **capping** (truncamento nos limites), que preserva os dados.

In [ ]:
# =============================================================================
# CÉLULA 12 — DETECÇÃO E TRATAMENTO DE OUTLIERS (CAPPING IQR)
# O que faz : calcula os limites inferior e superior pelo método IQR para cada
#             variável numérica e trunca (clipa) os valores fora desse intervalo.
# Resultado : tabela com quantidade e percentual de outliers por coluna, e
#             confirmação de que o capping foi aplicado nos 3 splits.
#             Valores extremos são truncados — não removidos — preservando dados.
# =============================================================================

# fnlwgt é peso amostral do Census (sem interpretação direta como feature).
# educational-num tem distribuição natural por graus — capping seria incorreto.
outlier_cols = [c for c in num_cols if c not in ('fnlwgt', 'educational-num')]

bounds = {}         # armazena (limite_inf, limite_sup) por coluna
outlier_report = [] # acumula relatório para exibição

for col in outlier_cols:
    Q1  = train[col].quantile(0.25)   # 1º quartil: 25% dos dados estão abaixo
    Q3  = train[col].quantile(0.75)   # 3º quartil: 75% dos dados estão abaixo
    IQR = Q3 - Q1                     # amplitude interquartil = Q3 - Q1

    # Limites de Tukey: valores além de 1.5×IQR são considerados outliers
    lo = Q1 - 1.5 * IQR
    hi = Q3 + 1.5 * IQR
    bounds[col] = (lo, hi)

    # Conta outliers: valores abaixo do limite inferior OU acima do superior
    n_out = ((train[col] < lo) | (train[col] > hi)).sum()
    pct   = n_out / len(train) * 100
    outlier_report.append({
        'coluna': col, 'outliers': n_out, '%': round(pct, 2),
        'limite_inf': round(lo, 2), 'limite_sup': round(hi, 2)
    })

print(pd.DataFrame(outlier_report).to_string(index=False))

# clip() trunca valores abaixo do lower para o lower e acima do upper para
# o upper. Limites calculados no treino são aplicados em todos os splits.
for df in [train, val, test]:
    for col, (lo, hi) in bounds.items():
        df[col] = df[col].clip(lower=lo, upper=hi)

print('\nCapping aplicado com sucesso!')

---
## 3. Engenharia de Atributos (Feature Engineering)

### 3.1 Codificação de Categorias

**Por que codificar?**  
Algoritmos matemáticos não entendem strings como `"Private"` ou `"Male"`. Precisamos converter para números.

Duas estratégias principais:
- **Label Encoding**: converte cada categoria para um inteiro (0, 1, 2...). Usado quando há ordem natural (ex: grau de escolaridade) ou em árvores de decisão.
- **One-Hot Encoding**: cria uma coluna binária por categoria. Evita que o modelo interprete um número maior como "mais importante".

Para este projeto, usaremos **Label Encoding** com `LabelEncoder` para simplicidade e compatibilidade com todos os modelos testados.

In [ ]:
# =============================================================================
# CÉLULA 13 — SEPARAÇÃO DE FEATURES E TARGET
# O que faz : divide cada split em X (features de entrada) e y (target binário).
#             Converte o target string ('>50K' / '<=50K') para inteiro (1 / 0).
# Resultado : 6 objetos criados (X_train, y_train, X_val, y_val, X_test, y_test)
#             e proporção da classe positiva impressa em cada split.
# =============================================================================

TARGET = 'income'  # nome da coluna alvo

# drop() remove a coluna target; .copy() evita SettingWithCopyWarning
X_train = train.drop(columns=[TARGET]).copy()
X_val   = val.drop(columns=[TARGET]).copy()
X_test  = test.drop(columns=[TARGET]).copy()

# Converte target para binário: 1 = ganha >50K, 0 = ganha <=50K
# astype(int) transforma True/False em 1/0
y_train = (train[TARGET] == '>50K').astype(int)
y_val   = (val[TARGET]   == '>50K').astype(int)
y_test  = (test[TARGET]  == '>50K').astype(int)

# Exibe proporção da classe positiva em cada split
# (devem ser similares — confirma que a divisão foi estratificada)
print(f'Classe 1 (>50K) no treino: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Classe 1 (>50K) no val:    {y_val.sum()} ({y_val.mean()*100:.1f}%)')
print(f'Classe 1 (>50K) no test:   {y_test.sum()} ({y_test.mean()*100:.1f}%)')

In [ ]:
# =============================================================================
# CÉLULA 14 — CODIFICAÇÃO DE VARIÁVEIS CATEGÓRICAS (LABEL ENCODING)
# O que faz : converte cada categoria string em um número inteiro único.
#             Ex: 'Private'=0, 'Self-emp'=1, 'Gov'=2.
#             O encoder é treinado no X_train e aplicado em val/test.
# Resultado : X_train, X_val e X_test sem colunas de texto — apenas números.
#             Categorias novas em val/test (não vistas no treino) recebem -1.
# =============================================================================

encoders = {}  # guarda o encoder de cada coluna para uso futuro (ex: deploy)

# Seleciona apenas colunas categóricas (tipo object = string)
cat_features = X_train.select_dtypes(include='object').columns.tolist()

for col in cat_features:
    le = LabelEncoder()

    # fit_transform: aprende as categorias do treino E já transforma o treino
    X_train[col] = le.fit_transform(X_train[col])
    encoders[col] = le  # salva para aplicar em val/test

    # Para val e test: usamos o mapeamento aprendido no treino.
    # Categorias novas (não vistas no treino) recebem -1 em vez de erro.
    for df in [X_val, X_test]:
        mapping = {v: i for i, v in enumerate(le.classes_)}
        df[col] = df[col].map(mapping).fillna(-1).astype(int)

print(f'Colunas codificadas: {cat_features}')
X_train.head(3)  # exibe as primeiras linhas já com valores numéricos

### 3.2 Dimensionamento de Características (Feature Scaling)

**Por que escalar?**  
`age` varia de 17 a 90. `fnlwgt` pode chegar a 1.000.000. Algoritmos baseados em distância (como Regressão Logística) são sensíveis a essa diferença de escala: uma feature com valores maiores domina o cálculo.

**StandardScaler** transforma cada feature para ter média 0 e desvio padrão 1:  
`z = (x - média) / desvio_padrão`

Árvores (Random Forest, Gradient Boosting) não precisam de scaling, mas aplicar não prejudica.

In [ ]:
# =============================================================================
# CÉLULA 15 — NORMALIZAÇÃO DAS FEATURES (STANDARDSCALER)
# O que faz : transforma cada feature para ter média 0 e desvio padrão 1,
#             usando a fórmula z = (x - média) / desvio_padrão.
# Resultado : arrays numpy X_train_scaled, X_val_scaled, X_test_scaled com
#             todas as features na mesma escala. Imprime o shape final.
#             Necessário para algoritmos sensíveis à escala (ex: Regressão
#             Logística, KNN, SVM). Não prejudica algoritmos de árvore.
# =============================================================================

scaler = StandardScaler()

# fit_transform no treino: calcula média e desvio do treino E normaliza
X_train_scaled = scaler.fit_transform(X_train)

# transform em val/test: usa a média e desvio calculados no treino
# NUNCA usar fit_transform aqui — isso seria data leakage
X_val_scaled  = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Salva os nomes das features para uso nos gráficos e relatórios
feature_names = X_train.columns.tolist()

print(f'Features após scaling: {X_train_scaled.shape[1]}')
print(f'Shape treino: {X_train_scaled.shape}')

### 3.3 Seleção de Atributos

**Por que selecionar features?**  
Nem toda variável contribui igualmente. Features irrelevantes adicionam ruído e aumentam o tempo de treino.

Usaremos **Informação Mútua** — uma medida estatística que captura dependências não-lineares entre cada feature e o target. Diferente da correlação de Pearson, ela detecta relações que não são lineares.

In [ ]:
# =============================================================================
# CÉLULA 16 — IMPORTÂNCIA POR INFORMAÇÃO MÚTUA
# O que faz : calcula o quanto cada feature compartilha informação com o target
#             usando a métrica de Informação Mútua (MI).
# Resultado : gráfico de barras horizontal salvo em 'fig_mutual_info.png' e
#             tabela ordenada por MI Score. Features com MI próximo de 0
#             contribuem pouco e são candidatas ao descarte.
#             MI detecta relações não-lineares (diferente da correlação de
#             Pearson, que só captura relações lineares).
# =============================================================================

# mutual_info_classif calcula o MI de cada feature em relação ao y_train
# random_state garante que estimativas aleatórias sejam reproduzíveis
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=RANDOM_STATE)

# Cria DataFrame para facilitar ordenação e visualização
mi_df = pd.DataFrame({'feature': feature_names, 'MI Score': mi_scores})
mi_df = mi_df.sort_values('MI Score', ascending=True)  # ascending=True para barh

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(mi_df['feature'], mi_df['MI Score'], color='#9C27B0')
ax.set_title('Informação Mútua — relevância de cada feature', fontsize=13)
ax.set_xlabel('MI Score')
plt.tight_layout()
plt.savefig('fig_mutual_info.png', dpi=120)
plt.show()

# Exibe a tabela completa ordenada do maior para o menor MI
print(mi_df.sort_values('MI Score', ascending=False).to_string(index=False))

In [ ]:
# =============================================================================
# CÉLULA 17 — SELEÇÃO DAS TOP K FEATURES (SelectKBest)
# O que faz : seleciona as 12 features com maior Informação Mútua e descarta
#             as 2 de menor relevância (geralmente fnlwgt e education).
# Resultado : X_train_sel, X_val_sel, X_test_sel com 12 colunas selecionadas,
#             e lista com os nomes das features mantidas.
#             Reduzir features diminui overfitting e acelera o treino.
# =============================================================================

K = 12  # número de features a manter (das 14 disponíveis)

# SelectKBest com mutual_info_classif: seleciona as K features de maior MI
selector = SelectKBest(mutual_info_classif, k=K)

# fit_transform no treino: aprende quais são as K melhores E já filtra
X_train_sel = selector.fit_transform(X_train_scaled, y_train)

# transform em val/test: aplica a mesma seleção aprendida no treino
X_val_sel  = selector.transform(X_val_scaled)
X_test_sel = selector.transform(X_test_scaled)

# Recupera os nomes das features selecionadas usando o índice booleano
selected_features = [feature_names[i] for i in selector.get_support(indices=True)]
print(f'Features selecionadas ({K}): {selected_features}')

---
## 4. Modelagem — Comparação de Algoritmos

Testaremos 4 algoritmos progressivamente mais complexos. Isso é uma boa prática: sempre comece simples (baseline) e vá escalando só se necessário.

| Algoritmo | Tipo | Vantagem |
|-----------|------|----------|
| Regressão Logística | Linear | Rápido, interpretável, bom baseline |
| Árvore de Decisão | Não-linear | Visual, fácil de explicar |
| Random Forest | Ensemble | Robusto, lida bem com outliers |
| Gradient Boosting | Ensemble | Alta performance, ganha competições |

In [ ]:
# =============================================================================
# CÉLULA 18 — TREINO E COMPARAÇÃO DOS MODELOS BASE
# O que faz : treina 4 algoritmos com configurações padrão, avalia cada um
#             no conjunto de validação e compara as métricas lado a lado.
# Resultado : tabela ordenada por F1 Score decrescente e print por modelo.
#             Identifica o algoritmo com melhor performance antes do tuning.
# =============================================================================

def avaliar_modelo(nome, model, X_tr, y_tr, X_v, y_v):
    """
    Treina o modelo em X_tr/y_tr e avalia em X_v/y_v.
    Retorna dicionário com 5 métricas arredondadas para 4 casas decimais.
    """
    model.fit(X_tr, y_tr)                       # treina o modelo
    pred = model.predict(X_v)                    # gera previsões binárias (0 ou 1)
    return {
        'Modelo':    nome,
        'Accuracy':  round(accuracy_score(y_v, pred), 4),
        'Precision': round(precision_score(y_v, pred), 4),
        'Recall':    round(recall_score(y_v, pred), 4),
        'F1 Score':  round(f1_score(y_v, pred), 4),           # MÉTRICA PRINCIPAL
        'AUC-ROC':   round(roc_auc_score(y_v,
                           model.predict_proba(X_v)[:,1]), 4)  # usa probabilidade
    }

# Lista de (nome, instância) — facilita iterar e armazenar os modelos treinados
modelos_base = [
    ('Regressão Logística',  LogisticRegression(max_iter=500, random_state=RANDOM_STATE)),
    ('Árvore de Decisão',    DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ('Random Forest',        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
    ('Gradient Boosting',    GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)),
]

resultados   = []   # acumula resultados para o DataFrame final
fitted_models = {}  # guarda modelos treinados para uso posterior (tuning table)

for nome, modelo in modelos_base:
    res = avaliar_modelo(nome, modelo, X_train_sel, y_train, X_val_sel, y_val)
    resultados.append(res)
    fitted_models[nome] = modelo
    print(f'{nome}: F1={res["F1 Score"]:.4f} | AUC={res["AUC-ROC"]:.4f}')

# Cria DataFrame e ordena pelo F1 Score (o mais importante)
results_df = pd.DataFrame(resultados)
results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# =============================================================================
# CÉLULA 19 — GRÁFICO COMPARATIVO DOS MODELOS
# O que faz : plota um gráfico de barras agrupadas com todas as métricas
#             de todos os modelos lado a lado para comparação visual.
# Resultado : gráfico salvo em 'fig_comparacao_modelos.png'.
#             A linha vermelha tracejada em 0.80 serve como referência de
#             performance mínima aceitável para problema de negócio.
# =============================================================================

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
x      = np.arange(len(metrics))  # posições no eixo X para cada métrica
width  = 0.2                       # largura de cada barra
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']  # uma cor por modelo

fig, ax = plt.subplots(figsize=(13, 6))

# Itera sobre os modelos: cada modelo ocupa um offset diferente no eixo X
for i, row in results_df.iterrows():
    vals = [row[m] for m in metrics]         # extrai os valores das métricas
    ax.bar(x + i*width, vals, width,          # posição = x + deslocamento do modelo
           label=row['Modelo'], color=colors[i])

# Centraliza os ticks no meio do grupo de barras
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_title('Comparação de modelos — validação', fontsize=13)
ax.set_ylabel('Score')
ax.legend(loc='lower right')

# Linha de referência visual: modelos abaixo de 0.80 em F1 precisam de tuning
ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Referência 0.80')

plt.tight_layout()
plt.savefig('fig_comparacao_modelos.png', dpi=120)
plt.show()

---
## 5. Tuning de Hiperparâmetros

O **Gradient Boosting** costuma ter o melhor F1 Score neste dataset. Vamos otimizá-lo.

**O que são hiperparâmetros?**  
São configurações do algoritmo que você define ANTES do treino. O modelo aprende os parâmetros internos (pesos), mas os hiperparâmetros são escolha sua.

**RandomizedSearchCV** testa combinações aleatórias de hiperparâmetros e usa validação cruzada — mais eficiente que GridSearch quando o espaço é grande.

**Tabela de tuning** — registramos cada mudança para rastrear evolução:

In [ ]:
# =============================================================================
# CÉLULA 20 — TUNING DE HIPERPARÂMETROS (RandomizedSearchCV)
# O que faz : define um grid de hiperparâmetros para o Gradient Boosting e
#             testa 40 combinações aleatórias com validação cruzada 5-fold,
#             otimizando o F1 Score.
# Resultado : imprime os melhores hiperparâmetros encontrados.
#             best_gb = modelo final treinado com a melhor combinação.
#             Tempo estimado: 2 a 5 minutos dependendo da máquina.
# =============================================================================

# Espaço de busca: cada chave é um hiperparâmetro, cada lista são os valores
# possíveis que serão amostrados aleatoriamente.
param_grid = {
    'n_estimators':      [100, 200, 300, 400],   # número de árvores no ensemble
    'learning_rate':     [0.01, 0.05, 0.1, 0.2], # taxa de aprendizado (passo de cada árvore)
    'max_depth':         [3, 4, 5, 6],            # profundidade máxima de cada árvore
    'min_samples_split': [2, 5, 10],              # mínimo de amostras para dividir um nó
    'subsample':         [0.7, 0.8, 0.9, 1.0],   # fração de amostras usada por árvore
    'max_features':      ['sqrt', 'log2', None],  # features consideradas em cada split
}

gb_base = GradientBoostingClassifier(random_state=RANDOM_STATE)

# RandomizedSearchCV:
#   n_iter=40  → testa 40 combinações (não todas — mais eficiente que GridSearch)
#   cv=5       → valida cada combinação em 5 folds (5-fold cross-validation)
#   scoring='f1' → otimiza pelo F1 Score (nossa métrica principal)
#   n_jobs=-1  → usa todos os núcleos disponíveis do processador
search = RandomizedSearchCV(
    gb_base, param_grid,
    n_iter=40,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1   # exibe progresso da busca
)

# Treina com os dados de treino — val/test NÃO são vistos aqui
search.fit(X_train_sel, y_train)

# best_estimator_ = modelo já retreinado com a melhor combinação encontrada
best_gb     = search.best_estimator_
best_params = search.best_params_

print(f'\nMelhores hiperparâmetros:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# =============================================================================
# CÉLULA 21 — TABELA DE EVOLUÇÃO DO TUNING
# O que faz : registra o F1 Score em cada etapa do projeto, do baseline ao
#             modelo final otimizado, para documentar a evolução.
# Resultado : DataFrame com 4 linhas (uma por etapa) mostrando o ganho de
#             performance a cada decisão tomada.
# =============================================================================

# Gera previsões dos modelos já treinados para calcular o F1 na validação
gb_default_pred = fitted_models['Gradient Boosting'].predict(X_val_sel)
gb_tuned_pred   = best_gb.predict(X_val_sel)

tuning_table = pd.DataFrame([
    {
        'Etapa': '1 — Regressão Logística (baseline)',
        'Configuração': 'max_iter=500, padrão',
        # f1_score calcula o F1 da classe positiva (>50K = 1)
        'F1 Score Val': f1_score(y_val, fitted_models['Regressão Logística'].predict(X_val_sel)),
        'Observação': 'Ponto de partida — modelo simples e interpretável'
    },
    {
        'Etapa': '2 — Gradient Boosting (padrão)',
        'Configuração': 'n_estimators=100, lr=0.1, max_depth=3',
        'F1 Score Val': f1_score(y_val, gb_default_pred),
        'Observação': 'Melhora significativa sem tuning'
    },
    {
        'Etapa': '3 — GB + Feature Selection (K=12)',
        'Configuração': 'Remove 2 features de baixa MI',
        'F1 Score Val': f1_score(y_val, gb_default_pred),
        'Observação': 'Modelo mais enxuto sem perda de performance'
    },
    {
        'Etapa': '4 — GB + RandomizedSearchCV',
        'Configuração': str(best_params),
        'F1 Score Val': f1_score(y_val, gb_tuned_pred),
        'Observação': 'Melhor modelo — selecionado para avaliação final'
    },
])

# Arredonda F1 para 4 casas para facilitar leitura
tuning_table['F1 Score Val'] = tuning_table['F1 Score Val'].round(4)
tuning_table

---
## 6. XAI — Explicabilidade do Modelo (SHAP)

**O que é XAI?**  
XAI (Explainable AI) são técnicas para entender *por que* um modelo tomou uma decisão. Sem isso, temos uma "caixa-preta" — sabemos que funciona, mas não sabemos o motivo.

**SHAP (SHapley Additive exPlanations)** tem fundamento em teoria dos jogos. Para cada previsão, calcula quanto cada feature "contribuiu" para o resultado. Valores SHAP positivos puxam para `>50K`, negativos puxam para `<=50K`.

In [ ]:
# =============================================================================
# CÉLULA 22 — SHAP: IMPORTÂNCIA GLOBAL DAS FEATURES
# O que faz : calcula os valores SHAP para todas as observações do conjunto de
#             validação e plota a importância média absoluta por feature.
# Resultado : gráfico de barras salvo em 'fig_shap_importancia.png'.
#             Features com maior |SHAP| médio influenciam mais as previsões.
#             Diferente da importância nativa do sklearn, SHAP considera
#             interações entre features e é matematicamente justo.
# =============================================================================

# TreeExplainer é otimizado para modelos baseados em árvores (rápido e exato)
explainer = shap.TreeExplainer(best_gb)

# shap_values: array de shape (n_amostras, n_features)
# Cada valor = contribuição daquela feature para aquela previsão específica
shap_values = explainer.shap_values(X_val_sel)

# Calcula a importância média: média do valor absoluto por feature
# (positivo = empurra para >50K, negativo = empurra para <=50K)
shap_df   = pd.DataFrame(np.abs(shap_values), columns=selected_features)
mean_shap = shap_df.mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
mean_shap.sort_values().plot.barh(ax=ax, color='#FF5722')
ax.set_title('SHAP — Importância média de cada feature', fontsize=13)
ax.set_xlabel('|SHAP value| médio')
plt.tight_layout()
plt.savefig('fig_shap_importancia.png', dpi=120)
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 23 — SHAP: BEESWARM PLOT (IMPACTO POR OBSERVAÇÃO)
# O que faz : cria um gráfico "beeswarm" onde cada ponto representa uma
#             observação e sua posição no eixo X indica o impacto da feature
#             naquela previsão específica.
# Resultado : gráfico salvo em 'fig_shap_beeswarm.png'.
#             Cor dos pontos: azul = valor baixo da feature, vermelho = alto.
#             Posição X: à direita (+) = empurra para >50K,
#                        à esquerda (-) = empurra para <=50K.
#             Ex: capital-gain alto (vermelho, à direita) → mais provável >50K.
# =============================================================================

# shap.Explanation empacota os valores SHAP com metadados para os novos plots
shap_exp = shap.Explanation(
    values=shap_values,                   # os valores SHAP calculados
    base_values=explainer.expected_value, # valor médio base das previsões
    data=X_val_sel,                       # valores reais das features
    feature_names=selected_features       # nomes para exibir no eixo Y
)

plt.figure(figsize=(10, 7))
# max_display=12 limita às 12 features mais importantes para não poluir o gráfico
shap.plots.beeswarm(shap_exp, max_display=12, show=False)
plt.title('SHAP Beeswarm — impacto por observação', fontsize=13)
plt.tight_layout()
plt.savefig('fig_shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 24 — SHAP: WATERFALL PLOT (EXPLICAÇÃO LOCAL — UMA PREVISÃO)
# O que faz : explica passo a passo como o modelo chegou à previsão de UMA
#             pessoa específica, mostrando a contribuição de cada feature.
# Resultado : gráfico waterfall salvo em 'fig_shap_waterfall.png'.
#             Cada barra representa quanto uma feature aumentou (+, vermelho)
#             ou diminuiu (-, azul) a probabilidade de >50K.
#             O gráfico começa no valor base (média geral) e termina na
#             previsão final do modelo para aquela observação.
# =============================================================================

idx = 0  # índice da observação a explicar (0 = primeira linha do val)

# Gera a previsão binária para essa observação específica
pred_label = '>50K' if best_gb.predict(X_val_sel[[idx]])[0] == 1 else '<=50K'
print(f'Previsão do modelo para observação {idx}: {pred_label}')
print(f'Valor real (ground truth):               {">50K" if y_val.iloc[idx]==1 else "<=50K"}')

# waterfall[idx] acessa o objeto Explanation para a observação de índice idx
shap.plots.waterfall(shap_exp[idx], max_display=12, show=False)
plt.title(f'SHAP Waterfall — obs. {idx} | Previsão: {pred_label}', fontsize=12)
plt.tight_layout()
plt.savefig('fig_shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Avaliação Final — Conjunto de Teste

**REGRA DE OURO:** o test.csv só é usado UMA VEZ, no final. Usá-lo durante o tuning é "cola" — o modelo fica adaptado ao teste e os resultados não refletem performance real em dados novos. Isso se chama **data leakage**.

In [ ]:
# =============================================================================
# CÉLULA 25 — AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# O que faz : aplica o modelo otimizado (best_gb) no test.csv — usado pela
#             PRIMEIRA E ÚNICA VEZ — e calcula todas as métricas finais.
# Resultado : tabela de métricas com destaque para o F1 Score (métrica
#             principal do projeto). Estes são os resultados oficiais.
#             NUNCA reutilize o test.csv para ajustes — isso invalida o teste.
# =============================================================================

# predict() gera previsões binárias (0 ou 1) para cada linha do test
y_pred_test = best_gb.predict(X_test_sel)

# predict_proba() gera a probabilidade de cada classe: [:,1] = prob. de >50K
# Usado para calcular AUC-ROC (que precisa de probabilidade, não 0/1)
y_proba_test = best_gb.predict_proba(X_test_sel)[:, 1]

# Calcula todas as métricas comparando previsão com o valor real (y_test)
final_metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred_test),   # % de acertos totais
    'Precision': precision_score(y_test, y_pred_test),  # acurácia nos positivos previstos
    'Recall':    recall_score(y_test, y_pred_test),     # cobertura dos positivos reais
    'F1 Score':  f1_score(y_test, y_pred_test),         # equilíbrio entre precisão e recall
    'AUC-ROC':   roc_auc_score(y_test, y_proba_test),   # poder discriminativo geral
}

# Exibe as métricas formatadas com destaque no F1 Score
print('=' * 45)
print('       RESULTADO FINAL — TEST SET')
print('=' * 45)
for metric, value in final_metrics.items():
    marker = ' ◄ PRINCIPAL' if metric == 'F1 Score' else ''
    print(f'  {metric:<12}: {value:.4f}{marker}')
print('=' * 45)

In [ ]:
# =============================================================================
# CÉLULA 26 — MATRIZ DE CONFUSÃO
# O que faz : gera e visualiza a matriz de confusão do modelo no test set,
#             mostrando acertos e tipos de erro de forma explícita.
# Resultado : gráfico salvo em 'fig_confusion_matrix.png'.
#
#   Como ler a matriz:
#   ┌──────────────────┬────────────────┐
#   │ TN — previu <=50K│ FP — previu >50K│  ← real = <=50K
#   │   (acerto)       │  (falso alarme) │
#   ├──────────────────┼────────────────┤
#   │ FN — previu <=50K│ TP — previu >50K│  ← real = >50K
#   │  (perdeu o caso) │    (acerto)     │
#   └──────────────────┴────────────────┘
#   TN=True Negative, FP=False Positive, FN=False Negative, TP=True Positive
# =============================================================================

# confusion_matrix retorna array [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, y_pred_test)

fig, ax = plt.subplots(figsize=(6, 5))

# ConfusionMatrixDisplay formata a matriz com labels legíveis
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['<=50K', '>50K'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')  # azul = mais escuro = mais casos
ax.set_title('Matriz de Confusão — test set', fontsize=13)
plt.tight_layout()
plt.savefig('fig_confusion_matrix.png', dpi=120)
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 27 — RELATÓRIO DE CLASSIFICAÇÃO COMPLETO
# O que faz : imprime precisão, recall e F1 Score separados por classe,
#             além do suporte (número de amostras reais de cada classe).
# Resultado : tabela com métricas por classe (<=50K e >50K) e médias globais.
#             "macro avg" = média simples entre classes.
#             "weighted avg" = média ponderada pelo suporte de cada classe.
#             Permite ver se o modelo performa bem nas duas classes ou apenas
#             na majoritária (<=50K).
# =============================================================================

print(classification_report(y_test, y_pred_test,
                             target_names=['<=50K', '>50K']))

In [ ]:
# =============================================================================
# CÉLULA 28 — TABELA RESUMO FINAL DO PROJETO
# O que faz : consolida o F1 Score de cada etapa do projeto (validação e teste)
#             em uma única tabela para visualização do progresso total.
# Resultado : tabela com 4 linhas mostrando a evolução do F1 do baseline
#             até o modelo final, e o AUC-ROC final impresso separadamente.
#             Esta tabela é o resumo executivo do projeto.
# =============================================================================

summary = pd.DataFrame([
    {
        'Etapa': 'Baseline — Regressão Logística',
        'F1 (val)': round(f1_score(y_val,
                    fitted_models['Regressão Logística'].predict(X_val_sel)), 4),
        'F1 (test)': '—'  # não avaliado no test (não é o modelo final)
    },
    {
        'Etapa': 'Random Forest (padrão)',
        'F1 (val)': round(f1_score(y_val,
                    fitted_models['Random Forest'].predict(X_val_sel)), 4),
        'F1 (test)': '—'
    },
    {
        'Etapa': 'Gradient Boosting (padrão)',
        'F1 (val)': round(f1_score(y_val,
                    fitted_models['Gradient Boosting'].predict(X_val_sel)), 4),
        'F1 (test)': '—'
    },
    {
        'Etapa': '★ Gradient Boosting + Tuning (FINAL)',
        'F1 (val)': round(f1_score(y_val, best_gb.predict(X_val_sel)), 4),
        'F1 (test)': round(final_metrics['F1 Score'], 4)  # resultado oficial
    },
])

print('\n=== EVOLUÇÃO DO PROJETO ===')
print(summary.to_string(index=False))

# AUC-ROC impresso separadamente pois foi calculado com probabilidades
print(f'\nAUC-ROC final (test): {final_metrics["AUC-ROC"]:.4f}')

---
## 8. Conclusões e Aprendizados

### O que construímos
Um pipeline completo de Machine Learning para classificação de renda, cobrindo cada etapa do ciclo de vida de um modelo real.

### Principais decisões
1. **F1 Score como métrica principal**: o dataset é desbalanceado (~75% <=50K). Acurácia seria enganosa.
2. **Capping em vez de remoção de outliers**: preservamos dados raros que carregam informação real.
3. **Moda no treino, aplicada no val/test**: evita leakage de informação.
4. **Gradient Boosting como modelo final**: ensembles de árvores são estado da arte para dados tabulares.
5. **SHAP para explicabilidade**: transparência é obrigatória em decisões que afetam pessoas.

### Para ir além
- Testar **LightGBM** ou **XGBoost** (mais rápidos que o GradientBoostingClassifier do sklearn)
- Experimentar **class_weight='balanced'** para lidar melhor com desbalanceamento
- Aplicar **SMOTE** (oversampling sintético da classe minoritária)
- Usar **Optuna** para otimização de hiperparâmetros mais eficiente